# Day 8 — Chatbot
### AI that remembers your conversation!
---
**Model:** facebook/blenderbot-400M-distill

**The big difference from Day 7:**
- Day 7 = AI generates text (no memory)
- Day 8 = AI has a conversation (remembers everything you said)


---
## Step 1 — Install and load the chatbot


In [ ]:
!pip install transformers gradio -q
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging
import warnings
warnings.filterwarnings('ignore')
logging.set_verbosity_error()

MODEL = 'facebook/blenderbot-400M-distill'

print('Loading chatbot model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
print('Chatbot ready!')


---
## Step 2 — Create the chat function

> We store the conversation as a simple list.
> Every message (yours + bot) gets added to that list.
> The bot reads the whole list each time — that is its memory!


In [ ]:
def chat(user_message, history):
    '''
    user_message : what you typed
    history      : list of all messages so far
    '''
    # Add your message to history
    history.append({'role': 'user', 'content': user_message})

    # Build the full conversation text for the model
    # Each message separated by a special token
    conversation = tokenizer.sep_token.join(
        [m['content'] for m in history]
    )

    # Tokenize and generate reply
    inputs = tokenizer(
        conversation,
        return_tensors = 'pt',
        truncation     = True,
        max_length     = 128
    )
    output = model.generate(
        **inputs,
        max_new_tokens = 60
    )
    bot_reply = tokenizer.decode(output[0], skip_special_tokens=True)

    # Add bot reply to history
    history.append({'role': 'bot', 'content': bot_reply})

    return bot_reply, history

print('chat() function ready!')


---
## Step 3 — Send one message


In [ ]:
# Start with empty history
history = []

# Send first message
reply, history = chat('What is artificial intelligence?', history)

print('You :', 'What is artificial intelligence?')
print('Bot :', reply)


---
## Step 4 — See the memory in action!

> Tell the bot something personal.
> Then ask a follow-up — it should remember!


In [ ]:
# Fresh start
history = []

# Message 1 — tell it something
reply1, history = chat('My favourite sport is cricket.', history)
print('You : My favourite sport is cricket.')
print('Bot :', reply1)
print()

# Message 2 — ask a follow-up
reply2, history = chat('What sport did I just mention?', history)
print('You : What sport did I just mention?')
print('Bot :', reply2)
print()
print('The bot should remember cricket!')


---
## Step 5 — Have a full conversation


In [ ]:
# Fresh start
history = []

questions = [
    'Tell me about smart cities.',
    'What sensors are used in smart cities?',
    'How does AI help manage traffic?',
    'What are the benefits for citizens?',
]

print('Full conversation about Smart Cities:')
print('=' * 45)
print()

for i, question in enumerate(questions, 1):
    reply, history = chat(question, history)
    print(f'Turn {i}')
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


---
## Step 6 — Try your own topic!

> ✏️ Change the questions to anything you like!


In [ ]:
# ✏️ Change these questions!
history = []

my_questions = [
    'Tell me about machine learning.',
    'How is it different from regular programming?',
    'Can you give me a simple example?',
]

for i, question in enumerate(my_questions, 1):
    reply, history = chat(question, history)
    print(f'You : {question}')
    print(f'Bot : {reply}')
    print()


---
## Step 7 — Build a Gradio Chatbot App

> Type any message — bot replies and remembers!
> Click **Clear** to start a fresh conversation.


In [ ]:
import gradio as gr

# Store history between messages
memory = {'history': []}

def chat_app(user_message, chatbox):
    if not user_message.strip():
        return chatbox, ''

    # Get reply using our chat() function
    reply, memory['history'] = chat(user_message, memory['history'])

    # Add to display
    chatbox.append((user_message, reply))
    return chatbox, ''


def clear_chat():
    # Reset memory — fresh start
    memory['history'] = []
    return [], ''


with gr.Blocks(title='AI Chatbot') as app:
    gr.Markdown('# AI Chatbot — Blenderbot')
    gr.Markdown('Chat with AI! It **remembers** your whole conversation.')
    gr.Markdown('Click **Clear** to start fresh.')

    chatbox   = gr.Chatbot(label='Conversation', height=350)
    msg_input = gr.Textbox(
        label       = 'Your message',
        placeholder = 'Type anything...',
        lines       = 1
    )
    with gr.Row():
        send_btn  = gr.Button('Send',  variant='primary')
        clear_btn = gr.Button('Clear')

    send_btn.click(
        fn      = chat_app,
        inputs  = [msg_input, chatbox],
        outputs = [chatbox, msg_input]
    )
    clear_btn.click(
        fn      = clear_chat,
        outputs = [chatbox, msg_input]
    )

app.launch(share=True)


In [ ]:
app.close()
print('App closed.')


---
## What we learned

| Step | What we did |
|------|-------------|
| 1 | Loaded Blenderbot model directly |
| 2 | Created chat() function with history list |
| 3 | Sent one message and got a reply |
| 4 | Saw memory in action |
| 5 | Had a full 4-turn conversation |
| 6 | Tried our own topic |
| 7 | Built Gradio chatbot app |

### Key words:
- **`history`** — a list that stores all messages
- **Memory** — passing the same history list to every message
- **Clear** — resetting history to empty list `[]`

**Tomorrow — Day 9:** Translation + Summarization!
